# Streaming Pipeline — Inserción directa en las 6 tablas Bronze

Este notebook implementa la **ingesta event-driven** de nuevos registros para
todas las entidades del Lakehouse Wanderbricks, cumpliendo la sección 3.1
de la guía del proyecto.

## Arquitectura

```
Productor sintético (PySpark) — UN solo notebook por simplicidad
         │ genera eventos para 6 entidades
         ▼
   INSERT INTO bronze.bronze_destinations  (sin FK)
   INSERT INTO bronze.bronze_users         (sin FK)
   INSERT INTO bronze.bronze_properties    (FK → destinations)
   INSERT INTO bronze.bronze_bookings      (FK → users, properties)
   INSERT INTO bronze.bronze_payments      (FK → bookings)
   INSERT INTO bronze.bronze_reviews       (FK → bookings, users, properties)
         │
         ▼
   dbt run --select silver    ← procesa TODO sin distinción de origen
         │
         ▼
   dbt run --select gold      ← Star Schema actualizado
         │
         ▼
   Power BI Dashboard refresca
```

## Decisiones de diseño

### ¿Por qué un solo notebook para las 6 entidades?

- **Demo coherente**: una sola ejecución llena toda la plataforma con eventos nuevos.
- **Orden correcto de FKs**: las inserciones siguen el orden topológico de
  dependencias (destinations → users → properties → bookings → payments → reviews)
  para evitar registros huérfanos.
- **Fácil de defender**: el profesor ve un único simulador que demuestra el
  patrón productivo.

### ¿Por qué insertar directo en cada `bronze.bronze_<entidad>` y no en tablas separadas?

- **Una entidad = una tabla** (patrón estándar de Snowflake, Iceberg, Lakehouse moderno).
- Evita el anti-patrón de tablas `_stream`, `_events`, `_topic` que complican el linaje.
- Silver/Gold no necesitan saber el origen: procesan la tabla unificada.

### ¿dbt necesita configuración extra para los nuevos registros?

**NO**. Los modelos en `dbt/models/silver/*.sql` ya leen con `{{ source('bronze', 'bronze_<entidad>') }}`.
La materialización `table` configurada en `dbt_project.yml` reconstruye Silver y Gold con
los registros actualizados en cada `dbt run`. **Cero configuración adicional**.

## 1. Configuración global

In [ ]:
# Catálogo actual (auto-detectado)
CATALOG_NAME = spark.sql("SELECT current_catalog()").collect()[0][0]

# Cantidad de eventos sintéticos a insertar por entidad
# Se mantiene proporción realista: muchas reservas, pocas propiedades/usuarios nuevos
N_DESTINATIONS = 5     # Nuevos destinos turísticos
N_USERS        = 30    # Nuevos usuarios registrados
N_PROPERTIES   = 10    # Nuevas propiedades publicadas
N_BOOKINGS     = 50    # Nuevas reservas (la entidad principal)
N_PAYMENTS     = 40    # Pagos asociados a las nuevas reservas
N_REVIEWS      = 20    # Reseñas de huéspedes recientes

print(f"Catálogo:        {CATALOG_NAME}")
print(f"Eventos a generar por entidad:")
print(f"  destinations:  {N_DESTINATIONS}")
print(f"  users:         {N_USERS}")
print(f"  properties:    {N_PROPERTIES}")
print(f"  bookings:      {N_BOOKINGS}")
print(f"  payments:      {N_PAYMENTS}")
print(f"  reviews:       {N_REVIEWS}")

## 2. Conteo inicial — antes de la simulación

Contamos los registros que ya existen para luego validar las nuevas inserciones.

In [ ]:
%sql
SELECT 'bronze_destinations' AS entidad, COUNT(*) AS antes FROM bronze.bronze_destinations
UNION ALL
SELECT 'bronze_users',        COUNT(*) FROM bronze.bronze_users
UNION ALL
SELECT 'bronze_properties',   COUNT(*) FROM bronze.bronze_properties
UNION ALL
SELECT 'bronze_bookings',     COUNT(*) FROM bronze.bronze_bookings
UNION ALL
SELECT 'bronze_payments',     COUNT(*) FROM bronze.bronze_payments
UNION ALL
SELECT 'bronze_reviews',      COUNT(*) FROM bronze.bronze_reviews
ORDER BY entidad;

## 3. Capturar IDs máximos actuales

Usamos `MAX(pk) + 1` para generar IDs nuevos sin colisión con los existentes.

In [ ]:
from pyspark.sql.functions import max as F_max, min as F_min

def max_id(tabla, columna):
    val = spark.table(tabla).agg(F_max(columna).alias("m")).collect()[0]["m"]
    return (val or 0) + 1

NEXT_DESTINATION_ID = max_id("bronze.bronze_destinations", "destination_id")
NEXT_USER_ID        = max_id("bronze.bronze_users",        "user_id")
NEXT_PROPERTY_ID    = max_id("bronze.bronze_properties",   "property_id")
NEXT_BOOKING_ID     = max_id("bronze.bronze_bookings",     "booking_id")
NEXT_PAYMENT_ID     = max_id("bronze.bronze_payments",     "payment_id")
NEXT_REVIEW_ID      = max_id("bronze.bronze_reviews",      "review_id")

print(f"Próximos IDs:")
print(f"  destination_id desde: {NEXT_DESTINATION_ID}")
print(f"  user_id        desde: {NEXT_USER_ID}")
print(f"  property_id    desde: {NEXT_PROPERTY_ID}")
print(f"  booking_id     desde: {NEXT_BOOKING_ID}")
print(f"  payment_id     desde: {NEXT_PAYMENT_ID}")
print(f"  review_id      desde: {NEXT_REVIEW_ID}")

---

## ENTIDAD 1/6 — `bronze_destinations` (sin dependencias)

Nuevos destinos turísticos. Es la primera entidad en insertarse porque otras
entidades dependen de ella (properties tiene FK a destinations).

In [ ]:
paises = ['Spain', 'Italy', 'Greece', 'Portugal', 'France', 'Croatia', 'Turkey', 'Mexico', 'Brazil', 'Argentina']

nuevos_destinations = spark.range(N_DESTINATIONS).selectExpr(
    f"CAST({NEXT_DESTINATION_ID} + id AS BIGINT) AS destination_id",
    f"concat('New Destination ', CAST({NEXT_DESTINATION_ID} + id AS STRING)) AS destination",
    f"element_at(array({','.join(repr(p) for p in paises)}), CAST(1 + rand() * {len(paises) - 1} AS INT)) AS country",
    "concat('Region ', CAST(rand() * 100 AS INT)) AS state_or_province",
    "concat('REG-', CAST(rand() * 999 AS INT)) AS state_or_province_code",
    "'Auto-generated destination via streaming simulation' AS description"
)

(nuevos_destinations.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.bronze_destinations"))

print(f"✓ Insertados {N_DESTINATIONS} destinos nuevos")
nuevos_destinations.show(5, truncate=False)

## ENTIDAD 2/6 — `bronze_users` (sin dependencias)

Nuevos usuarios registrados. También se inserta antes que properties/bookings
porque esas tienen FK a users.

In [ ]:
paises_users = ['Spain', 'United States', 'Germany', 'Brazil', 'Mexico', 'Colombia', 'Argentina', 'France', 'Italy', 'United Kingdom']

nuevos_users = spark.range(N_USERS).selectExpr(
    f"CAST({NEXT_USER_ID} + id AS BIGINT) AS user_id",
    f"concat('newuser', CAST({NEXT_USER_ID} + id AS STRING), '@example.com') AS email",
    f"concat('User ', CAST({NEXT_USER_ID} + id AS STRING)) AS name",
    f"element_at(array({','.join(repr(p) for p in paises_users)}), CAST(1 + rand() * {len(paises_users) - 1} AS INT)) AS country",
    "CASE WHEN rand() < 0.7 THEN 'individual' ELSE 'business' END AS user_type",
    "current_timestamp() AS created_at",
    "CASE WHEN rand() < 0.3 THEN true ELSE false END AS is_business",
    "CASE WHEN rand() < 0.3 THEN concat('Company ', CAST(rand() * 1000 AS INT)) ELSE NULL END AS company_name"
)

(nuevos_users.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.bronze_users"))

print(f"✓ Insertados {N_USERS} usuarios nuevos")
nuevos_users.show(5, truncate=False)

## ENTIDAD 3/6 — `bronze_properties` (FK → destinations)

Nuevas propiedades publicadas. Se asignan a destinos existentes (incluyendo
los recién creados) para mantener integridad referencial.

In [ ]:
# Rango de destination_id válidos (incluyendo los recién creados)
min_dest = spark.table("bronze.bronze_destinations").agg(F_min("destination_id")).collect()[0][0]
max_dest = spark.table("bronze.bronze_destinations").agg(F_max("destination_id")).collect()[0][0]

tipos_propiedad = ['Apartment', 'House', 'Villa', 'Hotel Room', 'Hostel', 'Cabin', 'Loft']

nuevas_properties = spark.range(N_PROPERTIES).selectExpr(
    f"CAST({NEXT_PROPERTY_ID} + id AS BIGINT) AS property_id",
    f"CAST(10000 + rand() * 20000 AS BIGINT) AS host_id",
    f"CAST({min_dest} + rand() * ({max_dest} - {min_dest}) AS BIGINT) AS destination_id",
    f"concat('Property ', CAST({NEXT_PROPERTY_ID} + id AS STRING)) AS title",
    "'Auto-generated property via streaming simulation' AS description",
    "ROUND(50 + rand() * 450, 2) AS base_price",
    f"element_at(array({','.join(repr(t) for t in tipos_propiedad)}), CAST(1 + rand() * {len(tipos_propiedad) - 1} AS INT)) AS property_type",
    "CAST(1 + rand() * 7 AS INT) AS max_guests",
    "CAST(1 + rand() * 4 AS INT) AS bedrooms",
    "CAST(1 + rand() * 2 AS INT) AS bathrooms",
    "-90 + rand() * 180 AS property_latitude",
    "-180 + rand() * 360 AS property_longitude",
    "current_date() AS created_at"
)

(nuevas_properties.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.bronze_properties"))

print(f"✓ Insertadas {N_PROPERTIES} propiedades nuevas (asociadas a destinos existentes)")
nuevas_properties.show(5, truncate=False)

## ENTIDAD 4/6 — `bronze_bookings` (FK → users, properties)

Nuevas reservas. Cada una vincula un usuario y una propiedad existentes
(incluyendo los recién creados).

In [ ]:
# Rangos válidos de FKs (incluyen los recién creados)
min_u = spark.table("bronze.bronze_users").agg(F_min("user_id")).collect()[0][0]
max_u = spark.table("bronze.bronze_users").agg(F_max("user_id")).collect()[0][0]
min_p = spark.table("bronze.bronze_properties").agg(F_min("property_id")).collect()[0][0]
max_p = spark.table("bronze.bronze_properties").agg(F_max("property_id")).collect()[0][0]

nuevos_bookings = spark.range(N_BOOKINGS).selectExpr(
    f"CAST({NEXT_BOOKING_ID} + id AS BIGINT) AS booking_id",
    f"CAST({min_u} + rand() * ({max_u} - {min_u}) AS BIGINT) AS user_id",
    f"CAST({min_p} + rand() * ({max_p} - {min_p}) AS BIGINT) AS property_id",
    "date_add(current_date(), CAST(rand() * 90 AS INT)) AS check_in",
    "date_add(current_date(), CAST(rand() * 90 AS INT) + CAST(1 + rand() * 13 AS INT)) AS check_out",
    "CAST(1 + rand() * 5 AS INT) AS guests_count",
    "ROUND(50 + rand() * 1450, 2) AS total_amount",
    "CASE WHEN rand() < 0.5 THEN 'confirmed' WHEN rand() < 0.8 THEN 'pending' ELSE 'cancelled' END AS status",
    "current_timestamp() AS created_at",
    "current_timestamp() AS updated_at"
)

(nuevos_bookings.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.bronze_bookings"))

print(f"✓ Insertadas {N_BOOKINGS} reservas nuevas")
nuevos_bookings.show(5, truncate=False)

## ENTIDAD 5/6 — `bronze_payments` (FK → bookings)

Pagos asociados a las nuevas reservas (~80% de las reservas generan pago).

In [ ]:
# Tomar booking_ids de las reservas recién insertadas (las últimas N_BOOKINGS)
recent_bookings = (
    spark.table("bronze.bronze_bookings")
         .orderBy("booking_id", ascending=False)
         .limit(N_BOOKINGS)
)
booking_ids_for_payments = [r.booking_id for r in recent_bookings.collect()[:N_PAYMENTS]]
bookings_csv = ",".join(str(b) for b in booking_ids_for_payments)

metodos_pago = ['credit_card', 'paypal', 'bank_transfer']

nuevos_payments = spark.range(N_PAYMENTS).selectExpr(
    f"CAST({NEXT_PAYMENT_ID} + id AS BIGINT) AS payment_id",
    f"CAST(element_at(array({bookings_csv}), CAST(1 + id AS INT)) AS BIGINT) AS booking_id",
    "ROUND(50 + rand() * 1450, 2) AS amount",
    f"element_at(array({','.join(repr(m) for m in metodos_pago)}), CAST(1 + rand() * {len(metodos_pago) - 1} AS INT)) AS payment_method",
    "CASE WHEN rand() < 0.8 THEN 'completed' WHEN rand() < 0.95 THEN 'pending' ELSE 'failed' END AS status",
    "current_timestamp() AS payment_date"
)

(nuevos_payments.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.bronze_payments"))

print(f"✓ Insertados {N_PAYMENTS} pagos nuevos (asociados a las reservas recién creadas)")
nuevos_payments.show(5, truncate=False)

## ENTIDAD 6/6 — `bronze_reviews` (FK → bookings, users, properties)

Reseñas de huéspedes asociadas a las reservas recién insertadas.

In [ ]:
# Las primeras N_REVIEWS reservas recién insertadas tendrán reseña
recent_bookings_full = recent_bookings.limit(N_REVIEWS).collect()
bookings_csv_rev = ",".join(str(r.booking_id) for r in recent_bookings_full)
users_csv_rev    = ",".join(str(r.user_id) for r in recent_bookings_full)
props_csv_rev    = ",".join(str(r.property_id) for r in recent_bookings_full)

comentarios = [
    "Great stay, would book again!",
    "Excellent location and service.",
    "Clean and comfortable property.",
    "Average experience, nothing special.",
    "Loved every minute of my visit.",
    "Host was very responsive.",
    "Property matched the description perfectly.",
    "Will definitely return."
]

nuevas_reviews = spark.range(N_REVIEWS).selectExpr(
    f"CAST({NEXT_REVIEW_ID} + id AS BIGINT) AS review_id",
    f"CAST(element_at(array({bookings_csv_rev}), CAST(1 + id AS INT)) AS BIGINT) AS booking_id",
    f"CAST(element_at(array({users_csv_rev}), CAST(1 + id AS INT)) AS BIGINT) AS user_id",
    f"CAST(element_at(array({props_csv_rev}), CAST(1 + id AS INT)) AS BIGINT) AS property_id",
    "ROUND(1 + rand() * 4, 1) AS rating",
    f"element_at(array({','.join(repr(c) for c in comentarios)}), CAST(1 + rand() * {len(comentarios) - 1} AS INT)) AS comment",
    "false AS is_deleted",
    "current_timestamp() AS created_at",
    "current_timestamp() AS updated_at"
)

(nuevas_reviews.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.bronze_reviews"))

print(f"✓ Insertadas {N_REVIEWS} reseñas nuevas")
nuevas_reviews.show(5, truncate=False)

## 4. Conteo final — después de la simulación

Validamos que las 6 tablas Bronze crecieron correctamente.

In [ ]:
%sql
SELECT 'bronze_destinations' AS entidad, COUNT(*) AS registros_despues FROM bronze.bronze_destinations
UNION ALL
SELECT 'bronze_users',        COUNT(*) FROM bronze.bronze_users
UNION ALL
SELECT 'bronze_properties',   COUNT(*) FROM bronze.bronze_properties
UNION ALL
SELECT 'bronze_bookings',     COUNT(*) FROM bronze.bronze_bookings
UNION ALL
SELECT 'bronze_payments',     COUNT(*) FROM bronze.bronze_payments
UNION ALL
SELECT 'bronze_reviews',      COUNT(*) FROM bronze.bronze_reviews
ORDER BY entidad;

## 5. Próximo paso — Propagar los eventos a Silver y Gold con dbt

Para que los dashboards de Power BI vean los nuevos eventos, ejecuta el
notebook **`08_run_dbt.ipynb`** que:

1. Corre `dbt run --select silver` → reconstruye las 6 tablas Silver con los nuevos registros.
2. Corre `dbt run --select gold`   → reconstruye Star Schema (1 fact + 4 dims).
3. Corre `dbt test`                → valida calidad de datos (30+ tests).

Después en Power BI: **Actualizar** → los KPIs (GMV, Reservas, Usuarios) reflejan
los nuevos eventos.

## Conclusión

Este notebook demuestra el **pipeline end-to-end completo** del proyecto:

```
50 nuevas reservas + 30 usuarios + 10 propiedades + 5 destinos + 40 pagos + 20 reseñas
    │
    ▼ (este notebook)
Bronze (6 tablas actualizadas con append)
    │
    ▼ (dbt run --select silver — notebook 08)
Silver (6 tablas limpias reprocesadas)
    │
    ▼ (dbt run --select gold — notebook 08)
Gold (Star Schema actualizado)
    │
    ▼ (Power BI refresh)
Dashboards con datos frescos
```

### Decisiones de diseño defendibles ante el docente

- **Un único notebook con 6 secciones**: orden topológico de FKs preservado (destinations → users → properties → bookings → payments → reviews). Una sola ejecución llena toda la plataforma.

- **Inserción directa en cada `bronze_<entidad>`**: patrón "una entidad = una tabla" estándar en arquitecturas Lakehouse modernas (Snowflake, Iceberg). Evita tablas intermedias que complican el linaje.

- **dbt sin configuración adicional**: los modelos `silver/*.sql` leen con `{{ source('bronze', 'bronze_<entidad>') }}` y la materialización `table` reconstruye todo en cada `dbt run`. Los nuevos eventos se procesan automáticamente.

- **Integridad referencial preservada**: cada entidad usa IDs dentro del rango válido (incluyendo los recién creados) para que los JOINs en Gold no produzcan huérfanos. Los tests `relationships` de dbt validan esto automáticamente.

- **Volumen proporcional realista**: muchas reservas (50) y pocas propiedades nuevas (10), reflejando la dinámica real de un marketplace donde el inventario crece más lento que el consumo.

### Para la demo en vivo (3 minutos)

1. **Conteo inicial** (celda 2): mostrar X registros actuales.
2. **Run all** del notebook 06 → inserta 155 eventos en total.
3. **Conteo final** (celda final): X + 155 registros.
4. **Notebook 08 `Run all`** → dbt procesa Silver y Gold.
5. **Power BI refresh** → KPIs suben.